# Additional run: missing model responses

Этот ноутбук собирает строки, где модель не дала валидного ответа (нет reasoning и нет final answer),
формирует единый датасет для допрогона и считает статистику по:
- experiment
- treatment
- incentive
- model


In [46]:
from pathlib import Path
import pandas as pd

# If run from thinking_expos/, BASE should be attitude/runs
BASE = Path('attitude/runs')
if not BASE.exists():
    BASE = Path('thinking_expos/attitude/runs')

patterns = ['**/*markup*.parquet', '**/*markup*.csv']
files = []
for pat in patterns:
    files.extend(BASE.glob(pat))
files = sorted(set(files))

print(f'BASE: {BASE}')
print(f'Markup files found: {len(files)}')
for f in files:
    print(' -', f.relative_to(BASE))


BASE: attitude/runs
Markup files found: 5
 - incentive_C_balanced_realism_50_reasoning_trace/responses_final_incentive_C_balanced_realism_50_reasoning_trace_markup.parquet
 - incentive_X/responses_final_X_markup.parquet
 - no_incentive_50_reasoning_trace/responses_final_no_incentive_50_reasoning_trace_markup.parquet
 - no_incentive_50_reasoning_trace_p2/responses_final_no_incentive_50_reasoning_trace_p2_markup.parquet
 - no_incentive_50_reasoning_trace_t1/responses_final_no_incentive_50_reasoning_trace_t1_markup.csv


In [47]:
def has_text(s: pd.Series) -> pd.Series:
    s = s.fillna('').astype(str).str.strip().str.lower()
    return ~s.isin(['', 'nan', 'none', 'null'])

def first_existing(df: pd.DataFrame, candidates: list[str], default='') -> pd.Series:
    present = [c for c in candidates if c in df.columns]
    if not present:
        return pd.Series([default] * len(df), index=df.index)

    out = df[present[0]].astype('string')
    for c in present[1:]:
        out = out.fillna(df[c].astype('string'))
    return out.fillna(default)


In [48]:
dfs = []
for f in files:
    f = Path(f)
    if f.suffix.lower() == '.csv':
        dfi = pd.read_csv(f)
    else:
        dfi = pd.read_parquet(f)

    dfi = dfi.copy()
    dfi['source_file'] = str(f.relative_to(BASE))

    # Keep columns required for filtering and rerun payload
    required_cols = [
        'REASONING', 'stage1_reasoning_text', 'stage2_reasoning_text', 'reasoning_trace_combined',
        'final_answer_text', 'model', 'experiment_name', 'incentive_key', 'Incentive', 'Treatment',
        'System prompt', 'User prompt', 'system_prompt', 'user_prompt',
        'run_source', 'row_id', 'id'
    ]
    for c in required_cols:
        if c not in dfi.columns:
            dfi[c] = None

    dfs.append(dfi)

all_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print('Total loaded rows:', len(all_df))
print('Total columns    :', len(all_df.columns))


Total loaded rows: 7000
Total columns    : 66


In [49]:
reasoning_present = (
    has_text(all_df['REASONING'])
    | has_text(all_df['stage1_reasoning_text'])
    | has_text(all_df['stage2_reasoning_text'])
    | has_text(all_df['reasoning_trace_combined'])
)
answer_present = has_text(all_df['final_answer_text'])

missing_df = all_df[~reasoning_present & ~answer_present].copy()

missing_df['experiment'] = first_existing(missing_df, ['experiment_name', 'experiment'])
missing_df['treatment'] = first_existing(missing_df, ['Treatment', 'treatment'])
missing_df['incentive'] = first_existing(missing_df, ['incentive_key', 'Incentive', 'incentive'])
missing_df['model_name'] = first_existing(missing_df, ['model'])
missing_df['system_prompt'] = first_existing(missing_df, ['System prompt', 'system_prompt'])
missing_df['user_prompt'] = first_existing(missing_df, ['User prompt', 'user_prompt'])

print('Missing rows (no reasoning + no final answer):', len(missing_df))
print('Rows without system prompt:', (missing_df['system_prompt'].astype(str).str.strip() == '').sum())
print('Rows without user prompt  :', (missing_df['user_prompt'].astype(str).str.strip() == '').sum())


Missing rows (no reasoning + no final answer): 613
Rows without system prompt: 0
Rows without user prompt  : 0


In [50]:
# Final rerun dataset: required fields + useful technical metadata
# IMPORTANT: keep every row from missing_df (including duplicates)
rerun_cols = [
    'experiment', 'treatment', 'incentive', 'model_name',
    'system_prompt', 'user_prompt',
    'source_file', 'run_source', 'row_id', 'id'
]

rerun_df = missing_df[rerun_cols].copy()
rerun_df = rerun_df.rename(columns={'model_name': 'model'})
rerun_df = rerun_df.reset_index(drop=True)
rerun_df['missing_row_uid'] = rerun_df.index.astype(str)

out_csv = Path('missing_rows_for_additional_run.csv')
out_parquet = Path('missing_rows_for_additional_run.parquet')
rerun_df.to_csv(out_csv, index=False)
rerun_df.to_parquet(out_parquet, index=False)

print('Saved:', out_csv)
print('Saved:', out_parquet)
print('Rows in rerun dataset (with duplicates):', len(rerun_df))
rerun_df.head(10)


Saved: missing_rows_for_additional_run.csv
Saved: missing_rows_for_additional_run.parquet
Rows in rerun dataset (with duplicates): 613


,experiment,treatment,incentive,model,system_prompt,user_prompt,source_file,run_source,row_id,id,missing_row_uid
0,Hassner_2023_Persuasive_and_Unpersuasive_Criti...,Treatment applied: low expected informational ...,C_balanced_realism,z-ai/glm-4.6,You are answering as a plausible survey respon...,Read the following vignette and question.\n\nV...,incentive_C_balanced_realism_50_reasoning_trac...,None,None,None,0
1,Hassner_2023_Persuasive_and_Unpersuasive_Criti...,Treatment applied: NONE (none (baseline contro...,C_balanced_realism,qwen/qwen3.5-9b,You are answering as a plausible survey respon...,Read the following vignette and question.\n\nV...,incentive_C_balanced_realism_50_reasoning_trac...,None,None,None,1
2,Hassner_2023_Persuasive_and_Unpersuasive_Criti...,Treatment applied: NONE (none (baseline contro...,C_balanced_realism,qwen/qwen3.5-9b,You are answering as a plausible survey respon...,Read the following vignette and question.\n\nV...,incentive_C_balanced_realism_50_reasoning_trac...,None,None,None,2
3,Hassner_2023_Persuasive_and_Unpersuasive_Criti...,Treatment applied: NONE (none (baseline contro...,C_balanced_realism,qwen/qwen3.5-9b,You are answering as a plausible survey respon...,Read the following vignette and question.\n\nV...,incentive_C_balanced_realism_50_reasoning_trac...,None,None,None,3
4,Hassner_2023_Persuasive_and_Unpersuasive_Criti...,Treatment applied: NONE (none (baseline contro...,C_balanced_realism,qwen/qwen3.5-9b,You are answering as a plausible survey respon...,Read the following vignette and question.\n\nV...,incentive_C_balanced_realism_50_reasoning_trac...,None,None,None,4
5,Hassner_2023_Persuasive_and_Unpersuasive_Criti...,Treatment applied: NONE (none (baseline contro...,C_balanced_realism,qwen/qwen3.5-9b,You are answering as a plausible survey respon...,Read the following vignette and question.\n\nV...,incentive_C_balanced_realism_50_reasoning_trac...,None,None,None,5
6,Hassner_2023_Persuasive_and_Unpersuasive_Criti...,Treatment applied: NONE (none (baseline contro...,C_balanced_realism,qwen/qwen3.5-9b,You are answering as a plausible survey respon...,Read the following vignette and question.\n\nV...,incentive_C_balanced_realism_50_reasoning_trac...,None,None,None,6
7,Hassner_2023_Persuasive_and_Unpersuasive_Criti...,Treatment applied: NONE (none (baseline contro...,C_balanced_realism,qwen/qwen3.5-9b,You are answering as a plausible survey respon...,Read the following vignette and question.\n\nV...,incentive_C_balanced_realism_50_reasoning_trac...,None,None,None,7
8,Hassner_2023_Persuasive_and_Unpersuasive_Criti...,Treatment applied: NONE (none (baseline contro...,C_balanced_realism,qwen/qwen3.5-9b,You are answering as a plausible survey respon...,Read the following vignette and question.\n\nV...,incentive_C_balanced_realism_50_reasoning_trac...,None,None,None,8
9,Hassner_2023_Persuasive_and_Unpersuasive_Criti...,Treatment applied: NONE (none (baseline contro...,C_balanced_realism,qwen/qwen3.5-9b,You are answering as a plausible survey respon...,Read the following vignette and question.\n\nV...,incentive_C_balanced_realism_50_reasoning_trac...,None,None,None,9


In [51]:
# Stats requested: experiment / treatment / incentive / model
stats = (
    rerun_df.groupby(['experiment', 'treatment', 'incentive', 'model'], dropna=False)
    .size()
    .reset_index(name='n_missing_rows')
    .sort_values('n_missing_rows', ascending=False)
    .reset_index(drop=True)
)

stats_csv = Path('missing_rows_stats_by_exp_treatment_incentive_model.csv')
stats.to_csv(stats_csv, index=False)

print('Saved:', stats_csv)
print('Groups in stats:', len(stats))
stats.head(30)


Saved: missing_rows_stats_by_exp_treatment_incentive_model.csv
Groups in stats: 23


,experiment,treatment,incentive,model,n_missing_rows
0,Savary_Goldsmith_2020_Unobserved_Altruism_Will...,Treatment applied: donor name is publicly reco...,Not provided,qwen/qwen3.5-9b,82
1,Savary_Goldsmith_2020_Unobserved_Altruism_Will...,Treatment applied: NONE (none (private donatio...,Not provided,qwen/qwen3.5-9b,82
2,Savary_Goldsmith_2020_Unobserved_Altruism_Will...,Treatment applied: public recognition brings s...,Not provided,qwen/qwen3.5-9b,76
3,Hassner_2023_Persuasive_and_Unpersuasive_Criti...,Treatment applied: NONE (none (baseline contro...,C_balanced_realism,qwen/qwen3.5-9b,39
4,Hassner_2023_Persuasive_and_Unpersuasive_Criti...,Treatment applied: extended interrogation dura...,C_balanced_realism,qwen/qwen3.5-9b,33
5,Hassner_2023_Persuasive_and_Unpersuasive_Criti...,Treatment applied: low expected informational ...,C_balanced_realism,qwen/qwen3.5-9b,30
6,Savary_Goldsmith_2020_Unobserved_Altruism_Will...,Treatment applied: donor name is publicly reco...,C_balanced_realism,qwen/qwen3.5-9b,27
7,Savary_Goldsmith_2020_Unobserved_Altruism_Will...,Treatment applied: NONE (none (private donatio...,C_balanced_realism,qwen/qwen3.5-9b,27
8,Savary_Goldsmith_2020_Unobserved_Altruism_Will...,Treatment applied: NONE (none (private donatio...,X_measurement_neutrality,qwen/qwen3.5-9b,27
9,Savary_Goldsmith_2020_Unobserved_Altruism_Will...,Treatment applied: public recognition brings s...,C_balanced_realism,qwen/qwen3.5-9b,26


## Rerun missing rows through models

Этот блок прогоняет `rerun_df` повторно через модели и сохраняет ответы.
Разбиение делается по группе `(model, temperature)`, как вы просили.

In [52]:
import os
import asyncio
from datetime import datetime, timezone
from typing import Any, Dict, List, Tuple

import numpy as np
from tqdm.auto import tqdm
from openai import AsyncOpenAI

# ---------- API config ----------
API_KEY = "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"
if not API_KEY:
    raise ValueError("Set OPENROUTER_API_KEY (or API_KEY) in environment before rerun.")

BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
client = AsyncOpenAI(api_key=API_KEY, base_url=BASE_URL)

# ---------- Rerun config ----------
# Если хотите один temperature на модель — заполните словарь.
# Если модели нет в словаре, берётся DEFAULT_TEMPERATURE.
MODEL_TEMPERATURES: Dict[str, float] = {
    # "moonshotai/kimi-k2-thinking": 0.01,
    # "z-ai/glm-4.6": 0.01,
    # "qwen/qwen3.5-9b": 0.01,
}
DEFAULT_TEMPERATURE = 0.01

MAX_CONCURRENCY_PER_GROUP = 8
MAX_RETRIES = 3
REQUEST_TIMEOUT_SEC = 180

RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT_DIR = Path(f"additional_rerun_{RUN_TS}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

rerun_requests = rerun_df.copy()
rerun_requests["temperature"] = rerun_requests["model"].map(MODEL_TEMPERATURES).fillna(DEFAULT_TEMPERATURE)
rerun_requests["temperature"] = rerun_requests["temperature"].astype(float)

print("Requests to run:", len(rerun_requests))
print("Unique model x temperature groups:", rerun_requests[["model", "temperature"]].drop_duplicates().shape[0])
rerun_requests[["model", "temperature"]].value_counts().reset_index(name="n_rows")

Requests to run: 613
Unique model x temperature groups: 2


,model,temperature,n_rows
0,qwen/qwen3.5-9b,0.01,611
1,z-ai/glm-4.6,0.01,2


In [53]:
def _safe_to_str(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    return str(x)


def _extract_text_from_completion(completion: Any) -> str:
    """Try to robustly extract assistant final text from OpenAI-compatible response."""
    try:
        c0 = completion.choices[0]
        msg = c0.message
        content = getattr(msg, "content", "")

        if isinstance(content, str):
            return content.strip()

        if isinstance(content, list):
            parts = []
            for part in content:
                if isinstance(part, dict):
                    txt = part.get("text") or part.get("content") or ""
                    if txt:
                        parts.append(str(txt))
                else:
                    txt = getattr(part, "text", "")
                    if txt:
                        parts.append(str(txt))
            return "\n".join([p for p in parts if p]).strip()

        txt = getattr(msg, "text", "")
        if txt:
            return _safe_to_str(txt).strip()
    except Exception:
        pass

    return ""


async def _one_request(row: pd.Series, semaphore: asyncio.Semaphore) -> Dict[str, Any]:
    async with semaphore:
        model = _safe_to_str(row["model"])
        temperature = float(row["temperature"])
        system_prompt = _safe_to_str(row["system_prompt"])
        user_prompt = _safe_to_str(row["user_prompt"])

        payload = {
            "model": model,
            "temperature": temperature,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        }

        last_error = ""
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                completion = await asyncio.wait_for(
                    client.chat.completions.create(**payload),
                    timeout=REQUEST_TIMEOUT_SEC,
                )
                final_text = _extract_text_from_completion(completion)

                usage = getattr(completion, "usage", None)
                usage_prompt_tokens = getattr(usage, "prompt_tokens", None) if usage else None
                usage_completion_tokens = getattr(usage, "completion_tokens", None) if usage else None
                usage_total_tokens = getattr(usage, "total_tokens", None) if usage else None

                return {
                    "ok": True,
                    "attempt": attempt,
                    "error": "",
                    "response_text": final_text,
                    "prompt_tokens": usage_prompt_tokens,
                    "completion_tokens": usage_completion_tokens,
                    "total_tokens": usage_total_tokens,
                }
            except Exception as e:
                last_error = f"{type(e).__name__}: {e}"
                if attempt < MAX_RETRIES:
                    await asyncio.sleep(min(2 ** attempt, 8))

        return {
            "ok": False,
            "attempt": MAX_RETRIES,
            "error": last_error,
            "response_text": "",
            "prompt_tokens": None,
            "completion_tokens": None,
            "total_tokens": None,
        }

In [54]:
def _norm_series(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.strip()


def _first_col(df: pd.DataFrame, candidates: List[str], default: str = "") -> pd.Series:
    present = [c for c in candidates if c in df.columns]
    if not present:
        return pd.Series([default] * len(df), index=df.index)
    out = df[present[0]].astype("string")
    for c in present[1:]:
        out = out.fillna(df[c].astype("string"))
    return out.fillna(default)


def prepare_rerun_groups(df_requests: pd.DataFrame) -> List[Tuple[Tuple[str, float], pd.DataFrame]]:
    grouped = list(df_requests.groupby(["model", "temperature"], dropna=False, sort=True))
    print(f"Groups prepared: {len(grouped)}")
    return grouped


async def run_one_group(model: str, temperature: float, g: pd.DataFrame) -> pd.DataFrame:
    print(f"\n=== Running group: model={model} | temperature={temperature} | rows={len(g)} ===")
    semaphore = asyncio.Semaphore(MAX_CONCURRENCY_PER_GROUP)

    rows = [row for _, row in g.iterrows()]

    async def _run_idx(idx: int, row: pd.Series):
        return idx, await _one_request(row, semaphore)

    tasks = [asyncio.create_task(_run_idx(i, row)) for i, row in enumerate(rows)]

    indexed_outputs = []
    pbar = tqdm(total=len(tasks), desc=f"requests {model} @ t={temperature}", leave=False)
    for fut in asyncio.as_completed(tasks):
        idx, result = await fut
        indexed_outputs.append((idx, result))
        pbar.update(1)
    pbar.close()

    indexed_outputs.sort(key=lambda x: x[0])
    group_outputs = [x[1] for x in indexed_outputs]

    g_out = g.copy().reset_index(drop=True)
    g_out = pd.concat([g_out, pd.DataFrame(group_outputs)], axis=1)
    g_out["run_timestamp_utc"] = RUN_TS

    model_tag = str(model).replace("/", "__")
    temp_tag = str(temperature).replace(".", "p")
    group_csv = OUT_DIR / f"group_model={model_tag}__temp={temp_tag}.csv"
    group_parquet = OUT_DIR / f"group_model={model_tag}__temp={temp_tag}.parquet"
    g_out.to_csv(group_csv, index=False)
    g_out.to_parquet(group_parquet, index=False)

    ok_n = int(g_out["ok"].sum())
    fail_n = int((~g_out["ok"]).sum())
    print(f"Saved group -> {group_csv.name} | ok={ok_n}, fail={fail_n}")
    return g_out


async def run_selected_groups(groups: List[Tuple[Tuple[str, float], pd.DataFrame]], group_indices: List[int] | None = None) -> pd.DataFrame:
    idxs = group_indices if group_indices is not None else list(range(len(groups)))
    collected = []

    groups_pbar = tqdm(idxs, desc="groups (model x temperature)")
    for i in groups_pbar:
        (model, temperature), g = groups[i]
        groups_pbar.set_postfix_str(f"{model} @ t={temperature} | n={len(g)}")
        out = await run_one_group(model, temperature, g)
        collected.append(out)

    if not collected:
        return pd.DataFrame()

    full_df = pd.concat(collected, ignore_index=True)
    full_csv = OUT_DIR / "additional_rerun_full_results.csv"
    full_parquet = OUT_DIR / "additional_rerun_full_results.parquet"
    full_df.to_csv(full_csv, index=False)
    full_df.to_parquet(full_parquet, index=False)

    summary = (
        full_df.groupby(["model", "temperature"], dropna=False)
        .agg(
            n_rows=("ok", "size"),
            n_ok=("ok", "sum"),
            n_fail=("ok", lambda s: int((~s).sum())),
        )
        .reset_index()
        .sort_values(["n_fail", "n_rows"], ascending=[False, False])
    )
    summary["success_rate"] = np.where(summary["n_rows"] > 0, summary["n_ok"] / summary["n_rows"], np.nan)
    summary_csv = OUT_DIR / "additional_rerun_summary_by_model_temperature.csv"
    summary.to_csv(summary_csv, index=False)

    print("\nSaved full results:", full_csv)
    print("Saved summary:", summary_csv)
    return full_df


def merge_results_into_experiment_files(results_df: pd.DataFrame, base_runs_dir: Path = BASE) -> pd.DataFrame:
    """Merge rerun results back into source experiment files preserving duplicates."""
    ok_df = results_df[results_df["ok"] & _norm_series(results_df["response_text"]).ne("")].copy()
    if ok_df.empty:
        print("No successful rerun rows to merge.")
        return pd.DataFrame()

    merge_report = []
    key_cols = ["__k_model", "__k_exp", "__k_treatment", "__k_incentive", "__k_sys", "__k_user"]

    for source_file, add_chunk in ok_df.groupby("source_file", dropna=False):
        source_path = Path(base_runs_dir) / str(source_file)
        if not source_path.exists():
            print(f"Skip missing source file: {source_path}")
            continue

        if source_path.suffix.lower() == ".csv":
            src = pd.read_csv(source_path)
        else:
            src = pd.read_parquet(source_path)

        src = src.copy()
        if "final_answer_text" not in src.columns:
            src["final_answer_text"] = ""

        src["__k_model"] = _norm_series(_first_col(src, ["model"]))
        src["__k_exp"] = _norm_series(_first_col(src, ["experiment_name", "experiment"]))
        src["__k_treatment"] = _norm_series(_first_col(src, ["Treatment", "treatment"]))
        src["__k_incentive"] = _norm_series(_first_col(src, ["incentive_key", "Incentive", "incentive"]))
        src["__k_sys"] = _norm_series(_first_col(src, ["System prompt", "system_prompt"]))
        src["__k_user"] = _norm_series(_first_col(src, ["User prompt", "user_prompt"]))

        add = add_chunk.copy()
        add["__k_model"] = _norm_series(add["model"])
        add["__k_exp"] = _norm_series(add["experiment"])
        add["__k_treatment"] = _norm_series(add["treatment"])
        add["__k_incentive"] = _norm_series(add["incentive"])
        add["__k_sys"] = _norm_series(add["system_prompt"])
        add["__k_user"] = _norm_series(add["user_prompt"])

        before_missing = int(_norm_series(src["final_answer_text"]).eq("").sum())

        # 1-to-1 matching with duplicates preserved: pair rows by key + sequence number within key.
        src_empty = src[_norm_series(src["final_answer_text"]).eq("")].copy()
        src_empty = src_empty.reset_index().rename(columns={"index": "__src_idx"})
        src_empty["__seq"] = src_empty.groupby(key_cols).cumcount()

        add = add.reset_index(drop=True)
        add["__add_idx"] = add.index
        add["__seq"] = add.groupby(key_cols).cumcount()

        pair = src_empty[["__src_idx"] + key_cols + ["__seq"]].merge(
            add[["__add_idx", "response_text", "run_timestamp_utc", "temperature"] + key_cols + ["__seq"]],
            on=key_cols + ["__seq"],
            how="inner",
        )

        updated = 0
        if not pair.empty:
            for _, r in pair.iterrows():
                idx = int(r["__src_idx"])
                src.at[idx, "final_answer_text"] = r["response_text"]
                src.at[idx, "additional_rerun_filled"] = True
                src.at[idx, "additional_rerun_ts"] = r["run_timestamp_utc"]
                src.at[idx, "additional_rerun_temperature"] = r["temperature"]
            updated = len(pair)

        matched_add_idx = set(pair["__add_idx"].tolist()) if not pair.empty else set()
        add_unmatched = add[~add["__add_idx"].isin(matched_add_idx)].copy()

        appended = 0
        if not add_unmatched.empty:
            append_df = pd.DataFrame({c: [None] * len(add_unmatched) for c in src.columns})
            if "model" in append_df.columns:
                append_df["model"] = add_unmatched["model"].values
            if "experiment_name" in append_df.columns:
                append_df["experiment_name"] = add_unmatched["experiment"].values
            if "Treatment" in append_df.columns:
                append_df["Treatment"] = add_unmatched["treatment"].values
            if "incentive_key" in append_df.columns:
                append_df["incentive_key"] = add_unmatched["incentive"].values
            if "System prompt" in append_df.columns:
                append_df["System prompt"] = add_unmatched["system_prompt"].values
            if "User prompt" in append_df.columns:
                append_df["User prompt"] = add_unmatched["user_prompt"].values

            append_df["final_answer_text"] = add_unmatched["response_text"].values
            append_df["additional_rerun_filled"] = True
            append_df["additional_rerun_ts"] = add_unmatched["run_timestamp_utc"].values
            append_df["additional_rerun_temperature"] = add_unmatched["temperature"].values

            src = pd.concat([src, append_df], ignore_index=True)
            appended = len(append_df)

        src = src.drop(columns=[c for c in key_cols if c in src.columns], errors="ignore")

        if source_path.suffix.lower() == ".csv":
            src.to_csv(source_path, index=False)
        else:
            src.to_parquet(source_path, index=False)

        after_missing = int(_norm_series(src["final_answer_text"]).eq("").sum())

        merge_report.append(
            {
                "source_file": str(source_file),
                "rows_in_source_after": len(src),
                "updated_existing_rows": int(updated),
                "appended_rows": int(appended),
                "missing_final_before": int(before_missing),
                "missing_final_after": int(after_missing),
            }
        )

        print(
            f"Merged -> {source_file}: updated={updated}, appended={appended}, "
            f"missing_final {before_missing} -> {after_missing}"
        )

    report_df = pd.DataFrame(merge_report)
    report_path = OUT_DIR / "merge_back_report.csv"
    report_df.to_csv(report_path, index=False)
    print("\nSaved merge report:", report_path)
    return report_df

In [55]:
# Group summary only: how many rows in each model x temperature group
rerun_groups = prepare_rerun_groups(rerun_requests)

group_sizes = pd.DataFrame(
    [
        {
            "group_idx": i,
            "model": m,
            "temperature": t,
            "n_rows": len(g),
        }
        for i, ((m, t), g) in enumerate(rerun_groups)
    ]
).sort_values(["n_rows", "model"], ascending=[False, True]).reset_index(drop=True)

print(f"Total groups: {len(group_sizes)}")
print(f"Total rows  : {int(group_sizes['n_rows'].sum())}")
group_sizes

Groups prepared: 2
Total groups: 2
Total rows  : 613


,group_idx,model,temperature,n_rows
0,0,qwen/qwen3.5-9b,0.01,611
1,1,z-ai/glm-4.6,0.01,2


In [56]:
# Run and merge (separate cell)
# Run all groups, or pass specific indices, e.g. group_indices=[0, 2]
rerun_results = await run_selected_groups(rerun_groups, group_indices=None)

print("\nTotal rows:", len(rerun_results))
print("OK rows   :", int(rerun_results["ok"].sum()) if len(rerun_results) else 0)
print("Failed    :", int((~rerun_results["ok"]).sum()) if len(rerun_results) else 0)

# Merge successful additional responses back into original experiment files
merge_report = merge_results_into_experiment_files(rerun_results, base_runs_dir=BASE)
merge_report

groups (model x temperature):   0%|          | 0/2 [00:00<?, ?it/s]


=== Running group: model=qwen/qwen3.5-9b | temperature=0.01 | rows=611 ===


requests qwen/qwen3.5-9b @ t=0.01:   0%|          | 0/611 [00:00<?, ?it/s]

Saved group -> group_model=qwen__qwen3.5-9b__temp=0p01.csv | ok=452, fail=159

=== Running group: model=z-ai/glm-4.6 | temperature=0.01 | rows=2 ===


requests z-ai/glm-4.6 @ t=0.01:   0%|          | 0/2 [00:00<?, ?it/s]

Saved group -> group_model=z-ai__glm-4.6__temp=0p01.csv | ok=2, fail=0

Saved full results: additional_rerun_20260425T201559Z/additional_rerun_full_results.csv
Saved summary: additional_rerun_20260425T201559Z/additional_rerun_summary_by_model_temperature.csv

Total rows: 613
OK rows   : 454
Failed    : 159
Merged -> incentive_C_balanced_realism_50_reasoning_trace/responses_final_incentive_C_balanced_realism_50_reasoning_trace_markup.parquet: updated=168, appended=0, missing_final 207 -> 39
Merged -> incentive_X/responses_final_X_markup.parquet: updated=96, appended=0, missing_final 123 -> 27
Merged -> no_incentive_50_reasoning_trace/responses_final_no_incentive_50_reasoning_trace_markup.parquet: updated=83, appended=0, missing_final 135 -> 52
Merged -> no_incentive_50_reasoning_trace_p2/responses_final_no_incentive_50_reasoning_trace_p2_markup.parquet: updated=94, appended=0, missing_final 146 -> 52
Merged -> no_incentive_50_reasoning_trace_t1/responses_final_no_incentive_50_reasoning_

,source_file,rows_in_source_after,updated_existing_rows,appended_rows,missing_final_before,missing_final_after
0,incentive_C_balanced_realism_50_reasoning_trac...,1400,168,0,207,39
1,incentive_X/responses_final_X_markup.parquet,1400,96,0,123,27
2,no_incentive_50_reasoning_trace/responses_fina...,1400,83,0,135,52
3,no_incentive_50_reasoning_trace_p2/responses_f...,1400,94,0,146,52
4,no_incentive_50_reasoning_trace_t1/responses_f...,1400,2,0,2,0
